In [1]:
import torch
import random
import sys
import os
import re
import numpy as np


In [2]:
Skf_File = os.path.join("../corrupt_pp", "C-C.skf")
Result_File = os.path.join("..","detailed.out")

N_SAMPLES=10
LOWER = -0.6
UPPER = 0.0

In [3]:
def modify_skf(skf_file, output_file, Ep, Es, SPE):
    with open(skf_file, "r") as file:
            lines = file.readlines()
    line = lines[1]
    parameters = list(re.finditer( r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?", line ))
    replacements= [f"{Ep:.10f}", f"{Es:.10f}", f"{SPE:.10f}"]

    for parameter, replacement in reversed(list(zip(parameters[1:4], replacements))): 
          start, end = parameter.span()
          line = line[:start] + replacement + line[end:]
    lines[1] = line
    print("New Random ", Ep, ", ", Es, ", ",SPE)    
    with open(output_file, "w") as f:
          f.writelines(lines)

In [4]:
import dataset
input = []
output = []
for i in range(N_SAMPLES):
    Ep = random.uniform(LOWER, UPPER)
    Es = random.uniform(LOWER, UPPER)
    SPE = random.uniform(LOWER, UPPER)
    modify_skf(Skf_File, Skf_File, Ep, Es, SPE)
    energy, force = dataset.test_md()
    input_data = np.array([Ep, Es, SPE])    
    input.append(input_data)
    output_data = np.array([energy, force])
    output.append(output_data)
input_tensor = torch.tensor(input, dtype=torch.float64)
output_tensor = torch.tensor(output, dtype=torch.float64)


    

New Random  -0.3260721225283733 ,  -0.23721271140462452 ,  -0.14080356142846456
Force RMSE: 0.654520 eV/Å
Energy RMSE: 3004.080559 eV
New Random  -0.3463240947939758 ,  -0.5149296711870902 ,  -0.26307835224188475
Force RMSE: 0.269966 eV/Å
Energy RMSE: 2985.766378 eV
New Random  -0.24749895526707083 ,  -0.17771933561050174 ,  -0.2580137318350254
Force RMSE: 0.874162 eV/Å
Energy RMSE: 3022.343777 eV
New Random  -0.5553834988952597 ,  -0.22882257326350153 ,  -0.05877967000260609
Force RMSE: 1.582203 eV/Å
Energy RMSE: 2934.496348 eV
New Random  -0.3265423171253678 ,  -0.12286072385266855 ,  -0.3614441788933812
Force RMSE: 0.773662 eV/Å
Energy RMSE: 3007.404854 eV
New Random  -0.5436737104885256 ,  -0.590534989318059 ,  -0.579458451122763
Force RMSE: 1.916336 eV/Å
Energy RMSE: 2923.116629 eV
New Random  -0.39582434450606285 ,  -0.5442697799031679 ,  -0.4860218155009981
Force RMSE: 0.579475 eV/Å
Energy RMSE: 2971.604340 eV
New Random  -0.34774592091320705 ,  -0.4692325104057838 ,  -0.3473052

/tmp/ipykernel_682911/285206072.py:14: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:249.)
  input_tensor = torch.tensor(input, dtype=torch.float64)


In [5]:
print("Input Tensor: ", input_tensor)
print("Output Tensor: ", output_tensor)


Input Tensor:  tensor([[-0.3261, -0.2372, -0.1408],
        [-0.3463, -0.5149, -0.2631],
        [-0.2475, -0.1777, -0.2580],
        [-0.5554, -0.2288, -0.0588],
        [-0.3265, -0.1229, -0.3614],
        [-0.5437, -0.5905, -0.5795],
        [-0.3958, -0.5443, -0.4860],
        [-0.3477, -0.4692, -0.3473],
        [-0.1461, -0.1886, -0.1254],
        [-0.3942, -0.3946, -0.3954]], dtype=torch.float64)
Output Tensor:  tensor([[3.0041e+03, 6.5452e-01],
        [2.9858e+03, 2.6997e-01],
        [3.0223e+03, 8.7416e-01],
        [2.9345e+03, 1.5822e+00],
        [3.0074e+03, 7.7366e-01],
        [2.9231e+03, 1.9163e+00],
        [2.9716e+03, 5.7948e-01],
        [2.9884e+03, 2.3546e-01],
        [3.0392e+03, 1.1093e+00],
        [2.9810e+03, 5.5033e-01]], dtype=torch.float64)


In [6]:
# BO
bounds = torch.tensor([[LOWER, UPPER], [LOWER, UPPER], [LOWER, UPPER]], dtype=torch.float64)
def normalize(x):
    return (x - bounds[:, 0]) / (bounds[:, 1] - bounds[:, 0])
def denormalize(x):
    return x * (bounds[:, 1] - bounds[:, 0]) + bounds[:, 0]

train_x = normalize(input_tensor)


In [7]:
scale = torch.tensor([1.0, 1.0], dtype=torch.float64)

def calculate_loss(outputs):
    # Calculate the loss
    loss = torch.sum((outputs/scale) ** 2, dim=1, keepdim=True)
    return loss

L = calculate_loss(output_tensor)

train_obj = -L

In [8]:
import os
import torch
import warnings
from typing import Optional

from botorch import fit_gpytorch_mll
from botorch.optim import optimize_acqf

from botorch.models import SingleTaskGP, ModelListGP
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood
from botorch.acquisition.objective import GenericMCObjective
from botorch.exceptions import BadInitialCandidatesWarning
from botorch.models.transforms.input import Normalize

from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.acquisition import qLogExpectedImprovement, qLogNoisyExpectedImprovement

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.double
SMOKE_TEST = os.environ.get("SMOKE_TEST")

# build model
def generate_model(train_x, train_obj, state_dict = None):
    model_obj = SingleTaskGP(
        train_x,
        train_obj,
        input_transform=Normalize(d=train_x.shape[-1]),
        ).to(train_x)

    model = ModelListGP(model_obj)
    mll = SumMarginalLogLikelihood(model.likelihood, model)
    if state_dict is not None:
        model.load_state_dict(state_dict)
    print("model is generated")
    return mll, model

mll, model = generate_model(train_x, train_obj)


/home/basic-user/src/pyc/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model is generated
